# Projection Accuracy Analysis


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display, HTML

pd.set_option('display.float_format', '{:.2%}'.format)
pd.set_option('display.max_columns', 20)
sns.set_theme(style='whitegrid', palette='muted')

In [2]:
FILE_PATH = '../data/processed/projection_movie_50.csv'   # <-- change this to your file path

df = pd.read_csv(FILE_PATH)

# Normalise the match column: accept both boolean and string variants
if df['match'].dtype == object:
    df['match'] = df['match'].str.strip().str.lower().map({'true': True, 'false': False})
else:
    df['match'] = df['match'].astype(bool)

print(f"Rows: {len(df):,}  |  Columns: {list(df.columns)}")
df.head()

Rows: 66,000  |  Columns: ['m', 'col', 'model', 'row_id', 'match', 'true_value', 'projected_value', 'repetition']


,m,col,model,row_id,match,true_value,projected_value,repetition
0,1,actor_1,EmbDI_300,740,True,isaac_hempstead_wright,isaac_hempstead_wright,1
1,1,actor_1,EmbDI_300,942,True,viggo_mortensen,viggo_mortensen,1
2,1,actor_1,EmbDI_300,3899,True,armando_riesco,armando_riesco,1
3,1,actor_1,EmbDI_300,5411,True,marilyn_monroe,marilyn_monroe,1
4,1,actor_1,EmbDI_300,7602,True,ayesha_dharker,ayesha_dharker,1


In [3]:
print("=== Dataset Overview ===")
print(f"  Total predictions : {len(df):,}")
print(f"  Unique models     : {df['model'].nunique()} → {sorted(df['model'].unique())}")
print(f"  Unique columns    : {df['col'].nunique()} → {sorted(df['col'].unique())}")
print(f"  Repetitions       : {sorted(df['repetition'].unique())}")
print(f"  Overall accuracy  : {df['match'].mean():.2%}")
print()
print("Missing values:")
display(df.isnull().sum().rename('nulls').to_frame())

=== Dataset Overview ===
  Total predictions : 66,000
  Unique models     : 5 → ['EmbDI_300', 'EmbDI_512', 'HRR_1024', 'HRR_300', 'HRR_512']
  Unique columns    : 15 → ['actor_1', 'actor_2', 'actor_3', 'color', 'content_rating', 'director', 'genres', 'original_language', 'production_companies', 'production_countries', 'release_date_rounded', 'status', 'title', 'vote_average', 'year']
  Repetitions       : [0, 1, 2]
  Overall accuracy  : 88.84%

Missing values:


,nulls
m,0
col,0
model,0
row_id,0
match,0
true_value,15730
projected_value,13301
repetition,0


## Accuracy per Model x Column


In [4]:
def plot_model_col(d_frame, m_value, cols):

    model_list = [
    # "BSC_300",
    # "BSC_512",
    # "BSC_1024",
    # "BSC_2048",
    # "BSC_4096",
    # "BSC_8192",
    "EmbDI_300",
    "EmbDI_512",
    "HRR_300",
    "HRR_512",
    # "HRR_1024",
    # "HRR_2048",
    # "SemHDC_FastText_300"
    ]

    df_m = d_frame[(d_frame['m']== m_value) & (d_frame["model"].isin(model_list))]

    # Pivot: rows = model, columns = col
    acc_pivot = (
        df_m.groupby(['model', 'col'])['match']
        .agg(accuracy='mean', n_predictions='count', n_correct='sum')
        .reset_index()
    )

    pivot_table = acc_pivot.pivot(index='model', columns='col', values='accuracy')


    # Reorder rows to match model_list, keeping any unexpected models at the end
    ordered_models = [model for model in model_list if model in pivot_table.index]
    remaining_models = [model for model in pivot_table.index if model not in ordered_models]
    pivot_table = pivot_table.reindex(ordered_models + remaining_models)

    # Add row-level aggregates
    pivot_table.insert(0, 'OVERALL', df_m.groupby('model')['match'].mean())
    if len(cols) > 0:
        pivot_table = pivot_table[['OVERALL', *[c for c in cols if c != 'OVERALL']]]
    # pivot_table['BEST_COL']  = pivot_table.drop(columns='OVERALL').idxmax(axis=1)
    # pivot_table['WORST_COL'] = pivot_table.drop(columns=['OVERALL', 'BEST_COL']).idxmin(axis=1)

    # Colour the numeric cells
    numeric_cols = [c for c in pivot_table.columns if c not in ('BEST_COL', 'WORST_COL')]

    styled = (
        pivot_table.style
        .format({c: '{:.2%}' for c in numeric_cols})
        .background_gradient(cmap='RdYlGn', subset=numeric_cols, vmin=0, vmax=1)
        .set_caption(f'[m={m_value}] Accuracy by Model x Column')
        .set_table_styles([{'selector': 'caption',
                            'props': [('font-size', '14px'), ('font-weight', 'bold'),
                                    ('text-align', 'left'), ('padding-bottom', '6px')]}])
    )
    display(styled)
    print(pivot_table.round(2).to_latex(float_format="%.2f"))

In [5]:
# for m in sorted(df['m'].unique()):
plot_model_col(df.copy(), 4, [])
#     plot_model_col(df, m)

m = 15
plot_model_col(df.copy(), m, ['director', 'production_companies', 'content_rating', 'genres'])

col,OVERALL,actor_1,actor_2,actor_3,color
model,,,,,
EmbDI_300,93.00%,100.00%,100.00%,100.00%,72.00%
EmbDI_512,85.00%,100.00%,100.00%,100.00%,40.00%
HRR_300,100.00%,100.00%,100.00%,100.00%,100.00%
HRR_512,100.00%,100.00%,100.00%,100.00%,100.00%


\begin{tabular}{lrrrrr}
\toprule
col & OVERALL & actor_1 & actor_2 & actor_3 & color \\
model &  &  &  &  &  \\
\midrule
EmbDI_300 & 0.93 & 1.00 & 1.00 & 1.00 & 0.72 \\
EmbDI_512 & 0.85 & 1.00 & 1.00 & 1.00 & 0.40 \\
HRR_300 & 1.00 & 1.00 & 1.00 & 1.00 & 1.00 \\
HRR_512 & 1.00 & 1.00 & 1.00 & 1.00 & 1.00 \\
\bottomrule
\end{tabular}



col,OVERALL,director,production_companies,content_rating,genres
model,,,,,
EmbDI_300,78.27%,96.00%,68.00%,30.00%,82.00%
EmbDI_512,68.67%,96.00%,62.00%,8.00%,74.00%
HRR_300,50.13%,42.00%,51.33%,57.33%,48.00%
HRR_512,93.02%,92.67%,92.00%,98.00%,96.67%


\begin{tabular}{lrrrrr}
\toprule
col & OVERALL & director & production_companies & content_rating & genres \\
model &  &  &  &  &  \\
\midrule
EmbDI_300 & 0.78 & 0.96 & 0.68 & 0.30 & 0.82 \\
EmbDI_512 & 0.69 & 0.96 & 0.62 & 0.08 & 0.74 \\
HRR_300 & 0.50 & 0.42 & 0.51 & 0.57 & 0.48 \\
HRR_512 & 0.93 & 0.93 & 0.92 & 0.98 & 0.97 \\
\bottomrule
\end{tabular}

